In [ ]:
## Step 1: Import


import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt


---

## 📊 Step 2: Load Data


(x_train,_), (_,_)=keras.datasets.mnist.load_data()

x_train=x_train.astype("float32")/255.
x_train=np.reshape(x_train, (-1,28,28,1))


---

## 🎨 Step 3: Build Generator


latent_dim=100

generator=keras.Sequential([
layers.Dense(128,activation="relu",input_dim=latent_dim),
layers.Dense(28*28,activation="sigmoid"),
layers.Reshape((28,28,1))
])


---

## 🕵️ Step 4: Build Discriminator


discriminator=keras.Sequential([
layers.Flatten(input_shape=(28,28,1)),
layers.Dense(128,activation="relu"),
layers.Dense(1,activation="sigmoid")
])

discriminator.compile(
optimizer="adam",
loss="binary_crossentropy",
metrics=["accuracy"]
)


---

## 🔗 Step 5: Combined GAN


discriminator.trainable=False

gan_input=keras.Input(shape=(latent_dim,))
fake_image=generator(gan_input)
gan_output=discriminator(fake_image)

gan=keras.Model(gan_input,gan_output)

gan.compile(optimizer="adam",loss="binary_crossentropy")


---

## 🔁 Step 6: Training Loop


batch_size=128
epochs=10000

for epoch in range(epochs):

# ---------------------
# Train Discriminator
# ---------------------
idx=np.random.randint(0,x_train.shape[0],batch_size)
real_images=x_train[idx]

noise = np.random.normal(0,1, (batch_size,latent_dim))
fake_images=generator.predict(noise,verbose=0)

d_loss_real=discriminator.train_on_batch(real_images,np.ones((batch_size,1)))
d_loss_fake=discriminator.train_on_batch(fake_images,np.zeros((batch_size,1)))

# ---------------------
# Train Generator
# ---------------------
noise=np.random.normal(0,1, (batch_size,latent_dim))
g_loss=gan.train_on_batch(noise,np.ones((batch_size,1)))

# Print progress
ifepoch%1000==0:
print(f"Epoch{epoch} | D Loss:{d_loss_real[0]} | G Loss:{g_loss}")


---

## 🖼️ Step 7: Generate Images


noise=np.random.normal(0,1, (10,latent_dim))
generated_images=generator.predict(noise)

plt.figure(figsize=(10,2))
foriinrange(10):
plt.subplot(1,10,i+1)
plt.imshow(generated_images[i].reshape(28,28),cmap='gray')
plt.axis('off')
plt.show()